In [1]:
!pip install pdfplumber

   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.6 MB 4.2 MB/s eta 0:00:02
   ------- -------------------------------- 1.0/5.6 MB 2.2 MB/s eta 0:00:03
   ------- -------------------------------- 1.0/5.6 MB 2.2 MB/s eta 0:00:03
   -------------- ------------------------- 2.1/5.6 MB 2.4 MB/s eta 0:00:02
   ------------------ --------------------- 2.6/5.6 MB 2.7 MB/s eta 0:00:02
   -------------------------- ------------- 3.7/5.6 MB 2.9 MB/s eta 0:00:01
   ------------------------------- -------- 4.5/5.6 MB 3.2 MB/s eta 0:00:01
   ------------------------------------- -- 5.2/5.6 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 5.6/5.6 MB 3.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ------------- -------------------------- 1.0/3.1 MB 5.6 MB/s eta 0:00:01
   ----------------------- ------

In [7]:
# extract_pathway_table.py
# Requirements:
#   pip install pdfplumber pandas openpyxl tabulate
# Optional (better table extraction but requires Java):
#   pip install camelot-py[cv]
#
# Usage:
#   python extract_pathway_table.py
#
# Output:
#   pathway_evidence_table.csv
#   pathway_evidence_table.xlsx

import re
import pdfplumber
import pandas as pd
from pathlib import Path

PDF_PATH = Path("../drug_deepsearch_pdfs_final/Alpelisib.pdf")
CSV_OUT = Path("pathway_evidence_table.csv")
XLSX_OUT = Path("pathway_evidence_table.xlsx")

def split_row_by_spacing(line):
    """
    Split a single line into columns using two-or-more spaces as delimiter.
    Returns list of trimmed columns.
    """
    parts = re.split(r'\s{2,}', line.strip())
    return [p.strip() for p in parts if p.strip()]

def find_table_text_blocks(pdf):
    """
    Locate pages containing 'Pathway Evidence Table' or the table header and return
    concatenated text for contiguous block that looks like a table.
    """
    found_pages = []
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        if "Pathway Evidence Table" in text or "Pathway (ID/Name)" in text:
            found_pages.append(i)

    if not found_pages:
        # fallback: search for the table header text fragments
        for i, page in enumerate(pdf.pages):
            text = page.extract_text() or ""
            if "Pathway (ID/Name) Regulation" in text:
                found_pages.append(i)
    return found_pages

def simple_block_parse(text_block):
    """
    Heuristic parser: find header, then parse rows until a 'References' or end marker.
    It tries to merge multi-line cells: new row is assumed when a line contains
    a pattern that looks like a pathway identifier (e.g., 'KEGG', 'Reactome', 'KEGG:hsa', 'R-HSA')
    """
    lines = [ln for ln in (text_block.splitlines()) if ln.strip() != ""]
    # find header line index
    header_idx = None
    for idx, ln in enumerate(lines):
        if re.search(r'Pathway\s*\(ID/Name\)|Pathway \(ID/Name\)|Pathway Evidence Table', ln, re.I):
            header_idx = idx
            break
        if re.search(r'Pathway\s*\(ID/Name\)\s*Regulation', ln):
            header_idx = idx
            break
    if header_idx is None:
        # try to find the explicit column headings
        for idx, ln in enumerate(lines[:10]):
            if 'Regulation' in ln and 'Effect' in ln:
                header_idx = idx
                break

    start = header_idx + 1 if header_idx is not None else 0

    rows = []
    cur = None
    for ln in lines[start:]:
        if re.search(r'References?:', ln, re.I):
            break
        # detect beginning of a new row by presence of pathway-like token
        is_new = bool(re.search(r'(KEGG[:\s]|Reactome|R-HSA|hsa0|MSigDB|GO[:\s]|PI3K|PI3K–AKT|PI3K-AKT)', ln, re.I))
        # also treat a line as new row if it contains 'Sensitive' or 'Resistant' and at least one large whitespace-separated chunk
        if is_new:
            parts = split_row_by_spacing(ln)
            # best guess: map to 4-5 columns
            # if parts too short, pad; if longer, join extras into last column
            if len(parts) >= 4:
                pathway = parts[0]
                regulation = parts[1]
                effect = parts[2]
                rationale = "  ".join(parts[3:])
                rows.append([pathway, regulation, effect, rationale])
                cur = rows[-1]
            else:
                # store partial and expect continuation lines
                rows.append(parts)
                cur = rows[-1]
        else:
            # continuation of previous row: append to rationale (or last cell)
            if cur is None:
                # nothing to append to — skip or start new
                rows.append([ln])
                cur = rows[-1]
            else:
                # append text to last cell
                if len(cur) >= 4:
                    cur[3] = cur[3] + " " + ln.strip()
                else:
                    # just append as new last cell
                    cur.append(ln.strip())

    # normalize rows to 4 columns: Pathway, Regulation, Effect, Rationale/References
    normalized = []
    for r in rows:
        # flatten list into string segments if necessary
        if isinstance(r, str):
            r = [r]
        # join any list elements beyond 4 into the 4th column
        r = [str(x).strip() for x in r]
        if len(r) == 0:
            continue
        if len(r) == 1:
            # probably a lone line; skip
            continue
        if len(r) == 2:
            r = [r[0], r[1], "", ""]
        elif len(r) == 3:
            r = [r[0], r[1], r[2], ""]
        elif len(r) > 4:
            r = [r[0], r[1], r[2], " ".join(r[3:])]
        normalized.append(r[:4])
    return normalized

def extract_table(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        pages_with_table = find_table_text_blocks(pdf)
        if not pages_with_table:
            # best-effort: search all pages and concat the area that contains column names
            text = "\n\n".join([p.extract_text() or "" for p in pdf.pages])
            parsed = simple_block_parse(text)
            df = pd.DataFrame(parsed, columns=["Pathway (ID/Name)", "Regulation", "Effect on Alpelisib", "Rationale/References"])
            return df

        # try to extract tables from the located pages first
        collected_text = ""
        for pnum in pages_with_table:
            page = pdf.pages[pnum]
            # try structured table extraction
            try:
                tables = page.extract_tables()
            except Exception:
                tables = []
            if tables and len(tables) > 0:
                # choose the largest table (most rows)
                best = max(tables, key=lambda t: len(t))
                # convert to dataframe, cleaning empty columns
                df = pd.DataFrame(best[1:], columns=[c.strip() if c else f"col{i}" for i, c in enumerate(best[0])])
                # attempt to unify/rename likely columns
                rename_map = {}
                colnames = list(df.columns)
                for c in colnames:
                    if re.search(r'Pathway', c, re.I):
                        rename_map[c] = "Pathway (ID/Name)"
                    elif re.search(r'Regulation', c, re.I):
                        rename_map[c] = "Regulation"
                    elif re.search(r'Effect', c, re.I):
                        rename_map[c] = "Effect on Alpelisib"
                    elif re.search(r'Rationale', c, re.I):
                        rename_map[c] = "Rationale/References"
                df = df.rename(columns=rename_map)
                # ensure all target cols exist
                for col in ["Pathway (ID/Name)", "Regulation", "Effect on Alpelisib", "Rationale/References"]:
                    if col not in df.columns:
                        df[col] = ""
                return df[["Pathway (ID/Name)", "Regulation", "Effect on Alpelisib", "Rationale/References"]]

            # if no structured table, append text for heuristic parse
            collected_text += "\n\n" + (page.extract_text() or "")

        # heuristic parse the concatenated text region
        parsed_rows = simple_block_parse(collected_text)
        if not parsed_rows:
            # last fallback: try entire document text
            all_text = "\n\n".join([p.extract_text() or "" for p in pdf.pages])
            parsed_rows = simple_block_parse(all_text)

        df = pd.DataFrame(parsed_rows, columns=["Pathway (ID/Name)", "Regulation", "Effect on Alpelisib", "Rationale/References"])
        return df

def main():
    if not PDF_PATH.exists():
        raise FileNotFoundError(f"PDF not found at {PDF_PATH}")

    df = extract_table(PDF_PATH)
    # basic cleanup: strip whitespace, collapse internal whitespace, and deduplicate empty rows
    def clean_cell(x):
        if pd.isna(x):
            return ""
        s = re.sub(r'\s+', ' ', str(x)).strip()
        return s

    df = df.applymap(clean_cell)
    # drop rows that are almost empty
    df = df[df.apply(lambda row: any(len(str(v))>0 for v in row), axis=1)]
    df = df.reset_index(drop=True)

    # Save outputs
    df.to_csv(CSV_OUT, index=False)
    df.to_excel(XLSX_OUT, index=False)

    print("Extraction complete.")
    print(f"Saved CSV -> {CSV_OUT.resolve()}")
    print(f"Saved Excel -> {XLSX_OUT.resolve()}")
    print("\nPreview:")
    with pd.option_context('display.max_rows', 20, 'display.max_colwidth', 120):
        print(df.head(20).to_string(index=False))

if __name__ == "__main__":
    main()


Extraction complete.
Saved CSV -> D:\GS\PySingscore_replicate\singscore_new_code\results\deepsearch_tests\pathway_evidence_table.csv
Saved Excel -> D:\GS\PySingscore_replicate\singscore_new_code\results\deepsearch_tests\pathway_evidence_table.xlsx

Preview:
                                                                                                                                                                                                                                           Pathway (ID/Name) Regulation Effect on Alpelisib Rationale/References
Alpelisib directly inhibits PI3Kα, PI3K–AKT signaling blocking AKT activation and (KEGG:hsa04151; mTOR signaling, leading to Down Sensitive 3 4 Reactome R- reduced proliferation 3 . HSA-198203) PIK3CA mutations are driver mutations in sensitive tumors.                                                    
                                      Alpelisib-resistant cells show mTORC1 signaling elevated mTORC1 activity Up (in (Reactome R- R

C:\Users\Jeet\AppData\Local\Temp\ipykernel_19120\3739631114.py:202: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_cell)
